# B3b Defense – 03 Feature Engineering

**Objective:** Build `defense_feature_matrix.csv` from `macro_features_defense.csv`.
Target variable: FDEFX (absolute level, USD billions SAAR). Features include:
- Sparse lag features on ADEFNO absolute (lags 1,3,6,9,12,18,24) and ADEFNO_diff (lags 1,3),
  motivated by Defense Capital Goods order-to-shipment lead times of 3–24 months.
  Sparse selection avoids multicollinearity between consecutive lags while preserving coverage.
- Lag features on IPB52300S_diff (lags 1–6)
- Autoregressive lags on FDEFX absolute (lags 1–6); FDEFX_diff level already in raw features
- Rolling window mean statistics on FDEFX (3m, 6m, 12m)
- Calendar features (month, quarter, is_q4) — year excluded to prevent out-of-distribution extrapolation

In [1]:
import os
import pandas as pd
import numpy as np

In [2]:
DATA_PROCESSED = '../data/processed/'

# Load macro_features_defense.csv — explicit to_datetime() instead of parse_dates
# to avoid the slow dateutil fallback path in pandas 2.x for YYYY-MM strings.
input_path = os.path.join(DATA_PROCESSED, 'macro_features_defense.csv')
df = pd.read_csv(input_path)
df['date'] = pd.to_datetime(df['date'])
df.set_index('date', inplace=True)
df.sort_index(inplace=True)

print(f'Loaded: {input_path}')
print(f'Shape: {df.shape}')
print(f'Date range: {df.index.min().date()} to {df.index.max().date()}')
df.head()

Loaded: ../data/processed/macro_features_defense.csv
Shape: (311, 6)
Date range: 2000-02-01 to 2025-12-01


,ADEFNO,ADEFNO_diff,IPB52300S,IPB52300S_diff,FDEFX,FDEFX_diff
date,,,,,,
2000-02-01,5162,-1997.0,69.3458,-1.6863,383.028,0.000
2000-03-01,3773,-1389.0,68.6844,-0.6614,383.028,0.000
2000-04-01,3568,-205.0,67.3689,-1.3155,399.592,16.564
2000-05-01,5023,1455.0,66.5616,-0.8073,399.592,0.000
2000-06-01,25893,20870.0,67.1028,0.5412,399.592,0.000


In [3]:
# Target: FDEFX absolute (not differenced).
# Rationale: FDEFX measures realized federal defense expenditures, which is semantically
# consistent with B1/B2 invoice-based forecasts in SAC. Absolute values preserve
# interpretability — forecast outputs can be read directly in USD millions without
# cumulative reconstruction from diffs.
target = df['FDEFX'].copy()

# Initialize feature DataFrame starting from the raw data
feat = df.copy()

In [4]:
# ADEFNO as leading indicator for FDEFX.
# Sparse lag selection covers the full 3–24 month lead time range while avoiding
# multicollinearity between consecutive monthly lags:
#   lags 1,3     → short-term pipeline (small arms, ammunition: ~3–6 months)
#   lags 6,9     → medium-term pipeline (communications equipment: ~6–12 months)
#   lags 12,18,24 → long-term pipeline (aircraft, missiles: ~12–24 months)
# Two diff lags (1,3) capture short-term order momentum without redundancy.
ADEFNO_ABS_LAGS  = [1, 3, 6, 9, 12, 18, 24]
ADEFNO_DIFF_LAGS = [1, 3]

for lag in ADEFNO_ABS_LAGS:
    feat[f'ADEFNO_lag_{lag}'] = feat['ADEFNO'].shift(lag)

for lag in ADEFNO_DIFF_LAGS:
    feat[f'ADEFNO_diff_lag_{lag}'] = feat['ADEFNO_diff'].shift(lag)

print(f'ADEFNO absolute lags created: {ADEFNO_ABS_LAGS}')
print(f'ADEFNO diff lags created:     {ADEFNO_DIFF_LAGS}')

ADEFNO absolute lags created: [1, 3, 6, 9, 12, 18, 24]
ADEFNO diff lags created:     [1, 3]


In [5]:
# Lag features for IPB52300S_diff (lags 1–6)
# 6 months covers one half-year production cycle; lags 7–12 showed near-zero
# SHAP importance in prior analysis and are excluded.
for lag in range(1, 7):
    feat[f'IPB52300S_diff_lag_{lag}'] = feat['IPB52300S_diff'].shift(lag)

print('IPB52300S_diff lags 1–6 created.')

IPB52300S_diff lags 1–6 created.


In [6]:
# Autoregressive lags on FDEFX (target) — absolute only.
# SHAP analysis showed fdefx_lag_1/2/3 as the three strongest predictors overall.
# FDEFX_diff differenced lags are excluded: the raw FDEFX_diff column already
# carries 1-month momentum and individual diff lags showed negligible SHAP values.
for lag in range(1, 7):
    feat[f'fdefx_lag_{lag}'] = feat['FDEFX'].shift(lag)

print('FDEFX absolute AR lags 1–6 created.')

FDEFX absolute AR lags 1–6 created.


In [7]:
# Rolling window mean features on FDEFX absolute level (3m, 6m, 12m)
# shift(1) applied before rolling to prevent look-ahead leakage.
# Rolling std excluded: std features did not appear in the top-15 SHAP ranking
# and add noise without predictive benefit given the smoothness of FDEFX.
fdefx_shifted = feat['FDEFX'].shift(1)

feat['fdefx_rolling_3m_mean']  = fdefx_shifted.rolling(window=3).mean()
feat['fdefx_rolling_6m_mean']  = fdefx_shifted.rolling(window=6).mean()
feat['fdefx_rolling_12m_mean'] = fdefx_shifted.rolling(window=12).mean()

print('Rolling window mean features (3m, 6m, 12m) on FDEFX created with shift=1 to avoid leakage.')

Rolling window mean features (3m, 6m, 12m) on FDEFX created with shift=1 to avoid leakage.


In [8]:
# Calendar features derived from the DatetimeIndex
# year is excluded: as a numeric feature it causes out-of-distribution extrapolation
# for forecast year 2026 (outside training range 2002–2025), pulling predictions
# toward historical means rather than recent levels.
feat['month']   = feat.index.month
feat['quarter'] = feat.index.quarter
feat['is_q4']   = (feat['quarter'] == 4).astype(int)

print('Calendar features created: month, quarter, is_q4.')

Calendar features created: month, quarter, is_q4.


In [9]:
# Drop NaN rows introduced by lag and rolling operations
shape_before = feat.shape
feat.dropna(inplace=True)
shape_after = feat.shape

print(f'Shape before dropna: {shape_before}')
print(f'Shape after  dropna: {shape_after}')
print(f'Rows dropped: {shape_before[0] - shape_after[0]}')

Shape before dropna: (311, 33)
Shape after  dropna: (287, 33)
Rows dropped: 24


In [10]:
# Save defense_feature_matrix.csv to DATA_PROCESSED
out_path = os.path.join(DATA_PROCESSED, 'defense_feature_matrix.csv')
feat.to_csv(out_path)
print(f'Saved: {out_path}')
print()
print('head(3):')
print(feat.head(3).to_string())
print()
print('dtypes:')
print(feat.dtypes)

Saved: ../data/processed/defense_feature_matrix.csv

head(3):
            ADEFNO  ADEFNO_diff  IPB52300S  IPB52300S_diff    FDEFX  FDEFX_diff  ADEFNO_lag_1  ADEFNO_lag_3  ADEFNO_lag_6  ADEFNO_lag_9  ADEFNO_lag_12  ADEFNO_lag_18  ADEFNO_lag_24  ADEFNO_diff_lag_1  ADEFNO_diff_lag_3  IPB52300S_diff_lag_1  IPB52300S_diff_lag_2  IPB52300S_diff_lag_3  IPB52300S_diff_lag_4  IPB52300S_diff_lag_5  IPB52300S_diff_lag_6  fdefx_lag_1  fdefx_lag_2  fdefx_lag_3  fdefx_lag_4  fdefx_lag_5  fdefx_lag_6  fdefx_rolling_3m_mean  fdefx_rolling_6m_mean  fdefx_rolling_12m_mean  month  quarter  is_q4
date                                                                                                                                                                                                                                                                                                                                                                                                                            